# DocuMind PDF benchmark: retrieval stage only, on a Kaggle GPU

Scores extraction outputs produced elsewhere (the laptop run) through DocuMind's chunker, `bge-large-en-v1.5` and `bge-reranker-large`.

**Inputs:** the bundle dataset `documind-pdfbench-kaggle` (harness, app source, corpus) and the outputs dataset `documind-pdfbench-laptop-out` (`out/<extractor>/<doc>/` plus the driver script). **Settings:** GPU, Internet on. **Run:** Save & Run All. About 20 minutes. **Output:** `/kaggle/working/pdfbench_retrieval.zip` (`results/retrieval_scores.csv` + per-document cache).

In [ ]:
import os, shutil, zipfile
from pathlib import Path

ROOT, INPUT = Path("/tmp/documind"), Path("/kaggle/input")
os.environ.update({"PDFBENCH_ROOT": str(ROOT), "PDFBENCH_MODES": "baseline,structure_preserving"})

def find_tree():
    for run_py in INPUT.rglob("run.py"):
        if run_py.parent.name == "pdf_extraction" and (run_py.parents[2] / "data" / "benchmark_pdfs").is_dir():
            return run_py.parents[2]
    return None

def find_outputs():
    """The outputs dataset: <...>/benchmarks/pdf_extraction/out/<extractor>/<doc>/run.json, extracted or zipped."""
    roots = [INPUT]
    zips = [z for z in INPUT.rglob("*.zip") if "laptop-out" in z.name]
    if zips:
        dest = Path("/tmp/laptop_out"); dest.mkdir(exist_ok=True); zipfile.ZipFile(zips[0]).extractall(dest); roots.append(dest)
    for root in roots:
        for rj in root.rglob("run.json"):
            return rj.parents[2]  # .../out
    return None

if not (ROOT / "benchmarks" / "pdf_extraction" / "run.py").exists():
    tree = find_tree(); assert tree, "bundle dataset not attached"; shutil.copytree(tree, ROOT)
out = find_outputs(); assert out, "outputs dataset not attached"
os.environ["PDFBENCH_OUT"] = str(out)
script = out.parent / "kaggle" / "run_downstream_only.sh"   # shipped next to the outputs; newer than the bundle's copy
if script.exists():
    (ROOT / "benchmarks" / "pdf_extraction" / "kaggle").mkdir(parents=True, exist_ok=True)
    shutil.copy(script, ROOT / "benchmarks" / "pdf_extraction" / "kaggle" / "run_downstream_only.sh")
print("repo:", ROOT, "| outputs:", out, "| extractors:", sorted(p.name for p in out.iterdir() if p.is_dir()))

In [ ]:
!cd /tmp/documind && bash benchmarks/pdf_extraction/kaggle/run_downstream_only.sh 2>&1 | tee /kaggle/working/pdfbench_retrieval_run.log

In [ ]:
import csv, collections, os
p = "/tmp/documind/benchmarks/pdf_extraction/results/retrieval_scores.csv"
if os.path.exists(p):
    rows = list(csv.DictReader(open(p)))
    agg = collections.defaultdict(list)
    for r in rows: agg[(r["chunk_mode"], r["extractor"])].append(int(r["hit5"]))
    for (m, e), v in sorted(agg.items()): print(f"{m:22s} {e:13s} hit@5={sum(v)/len(v):.3f} n={len(v)}")
else:
    print("no retrieval_scores.csv; see pdfbench_retrieval_run.log")
z = "/kaggle/working/pdfbench_retrieval.zip"; print(z, f"{os.path.getsize(z)/1e6:.1f} MB" if os.path.exists(z) else "(missing)")